<a href="https://colab.research.google.com/github/bellatrix-ds/ml-in-crypto/blob/main/03_Smart_Contract_Usage_Clustering/deep_neural_autoencoders_based_clustering_of_evm_users.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📊 Data Handling
import pandas as pd
import numpy as np

# 📈 Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px



from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
import umap
import hdbscan
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('clustered_data.csv')

In [ ]:
# step 1
X = df.drop(columns=['Unnamed: 0', 'FROM_ADDRESS', 'dominant_category_encoded'], errors='ignore')
# step 2
X = X.fillna(0)

In [ ]:
# step 3: normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# step 4: Autoencoder
input_dim = X_scaled.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = layers.Dense(128, activation='relu')(input_layer)
encoded = layers.Dense(64, activation='relu')(encoded)
bottleneck = layers.Dense(32, activation='relu', name='embedding')(encoded)
decoded = layers.Dense(64, activation='relu')(bottleneck)
decoded = layers.Dense(128, activation='relu')(decoded)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
encoder = Model(inputs=input_layer, outputs=bottleneck)

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_scaled, X_scaled, epochs=50, batch_size=128, verbose=1)


In [ ]:
# step 5: embedding
X_embedded = encoder.predict(X_scaled)
nan_mask = ~np.isnan(X_embedded).any(axis=1)

X_embedded_clean = X_embedded[nan_mask]
features_clean = df[nan_mask].copy()

print(f"✅ تعداد داده بعد از حذف NaN: {X_embedded_clean.shape[0]} از {X_scaled.shape[0]}")


In [ ]:
# step 6: UMAP
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_embedded_clean)


In [ ]:
# step 7: clustering
clusterer = hdbscan.HDBSCAN(min_cluster_size=30, prediction_data=True)
labels = clusterer.fit_predict(X_umap)


In [ ]:
# step 8
features_clean['cluster'] = labels


In [ ]:
# step 9
plt.figure(figsize=(10, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=labels, cmap='Spectral', s=10)
plt.title("UMAP + HDBSCAN Clustering")
plt.colorbar(label='Cluster')
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.grid(True)
plt.show()

In [ ]:
# step 10
features_clean['cluster'].value_counts()

In [ ]:
cluster_summary = features_clean.groupby('cluster').agg({
    'category_diversity': 'mean',
    'value_sum_DeFi': 'mean',
    'value_sum_Governance': 'mean',
    'gas_efficiency_DeFi': 'mean',
    'cluster': 'count'
}).rename(columns={'cluster': 'user_count'}).sort_values('user_count', ascending=False)

display(cluster_summary)

In [ ]:
features_clean.info()

In [ ]:

cluster_summary = features_clean.groupby('cluster').agg({
    'category_diversity': 'mean',
    'value_sum_DeFi': 'mean',
    'value_sum_Governance': 'mean',
    'value_sum_Utility': 'mean',
    'gas_efficiency_DeFi': 'mean',
    'gas_efficiency_Governance': 'mean',
    'gas_efficiency_Utility': 'mean',
    'cluster': 'count'
}).rename(columns={
    'cluster': 'user_count'
}).sort_values('user_count', ascending=False)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_summary.drop(columns='user_count'), annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("📊 average in each cluster(Heatmap)")
plt.ylabel("Cluster ID")
plt.show()



In [ ]:
cluster_summary[['user_count']].head(20)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# فیلتر خوشه‌های با حداقل تعداد کاربر
cluster_summary_filtered = cluster_summary[cluster_summary['user_count'] > 50]

# حذف ستون user_count برای heatmap
data_for_heatmap = cluster_summary_filtered.drop(columns='user_count')

# ترسیم با تنظیمات بهتر
plt.figure(figsize=(14, 8))
sns.heatmap(data_for_heatmap, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Feature Mean"})

plt.title("🧠 Average behavior per cluster (Filtered & Enhanced Heatmap)", fontsize=16)
plt.xlabel("Feature", fontsize=12)
plt.ylabel("Cluster ID", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
cluster_summary_filtered.to_csv('result.csv')

In [ ]:
cluster_summary_filtered.to_excel('user_clusters.xlsx', index=False)
files.download('user_clusters.xlsx')